# Innovation Adoption (Bass Model)

*System Dynamics Studio, model library: Feedback*

This model explains how new products or ideas spread through a population. It identifies two main drivers: Innovation (people adopting because of advertising) and Imitation (people adopting because they see others using it). The famous "S-Curve" of adoption is generated by this dynamic, starting slowly and then accelerating rapidly as the "word of mouth" feedback loop takes over.

**Concepts:** Reinforcing loop, Balancing loop, S-curve

**Expected behavior:** S-shaped growth of adopters; the adoption rate rises then falls as the market saturates.

**Limitations:** Single product, fixed market size, no discarding or repeat purchase.

## Model structure

**Stock balances**

$$
\begin{aligned}
\frac{d\,\text{potential\_adopters}}{dt} &= - \text{adoption\_rate} \\
\frac{d\,\text{adopters}}{dt} &= \text{adoption\_rate} \\
\end{aligned}
$$

**Stocks**

| Stock | Initial value | Unit |
|---|---|---|
| `potential_adopters` | 10000 | — |
| `adopters` | 0 | — |

**Flows and auxiliaries** (in evaluation order)

$$
\begin{aligned}
\text{word\_of\_mouth} &= \text{potential\_adopters}\cdot\left(\frac{\text{adopters}}{\text{total\_population}}\right)\cdot\text{contact\_rate}\cdot\text{adoption\_fraction} \\
\text{sales\_from\_ads} &= \text{potential\_adopters}\cdot\text{ads\_effectiveness} \\
\text{adoption\_rate} &= \text{sales\_from\_ads}+\text{word\_of\_mouth} \\
\end{aligned}
$$

**Parameters**

| Parameter | Value | Unit |
|---|---|---|
| `total_population` | 10000 | — |
| `adoption_fraction` | 0.015 | — |
| `ads_effectiveness` | 0.015 | — |
| `contact_rate` | 100 | — |

## Setup

NumPy, SciPy and Matplotlib are preinstalled on Google Colab. The step, ramp and pulse helpers
are the time functions available in the app; they are defined here for completeness.

In [ ]:
# Generated by System Dynamics Studio
# Model time: 0 to 10 (time unit not set), output every 0.1
# In-app solver: RK4, fixed dt = 0.1.
# This script integrates with scipy.integrate.odeint (adaptive LSODA), so values
# match the app within numerical tolerance, not digit for digit.
import numpy as np
from scipy.integrate import odeint
import matplotlib.pyplot as plt

# Time functions (discontinuous; same definitions as in the app)
def sd_step(height, start, t):
    return np.where(t >= start, height, 0.0)

def sd_ramp(slope, start, *rest):
    *end, t = rest
    until = np.minimum(t, end[0]) if end else t
    return np.where(t > start, slope * (until - start), 0.0)

def sd_pulse(height, start, width, t):
    return np.where((t >= start) & (t < start + width), height, 0.0)

## Parameters and initial values

Edit the values below and rerun the notebook from here to try your own scenarios.

In [ ]:
# 1. Model Parameters (Constants)
params = {
    'total_population': 10000,
    'adoption_fraction': 0.015,
    'ads_effectiveness': 0.015,
    'contact_rate': 100
}

# 2. Initial State (Stocks)
y0 = [
    10000,  # potential_adopters
    0  # adopters
]

## Model equations

`model` returns the rate of change of every stock. It is the system of ordinary differential equations shown above.

In [ ]:
# 3. Model Definition
def model(y, t, p):
    # Unpack stocks
    potential_adopters, adopters = y

    # Auxiliaries & Flows (dependency order)
    word_of_mouth = (((potential_adopters * ((adopters / p['total_population']))) * p['contact_rate']) * p['adoption_fraction'])
    sales_from_ads = (potential_adopters * p['ads_effectiveness'])
    adoption_rate = (sales_from_ads + word_of_mouth)

    # Derivatives (Change in Stocks)
    dpotential_adopters_dt = - adoption_rate
    dadopters_dt = adoption_rate

    return [dpotential_adopters_dt, dadopters_dt]

## Simulation

The model runs from t = 0 to t = 10 time units. The app integrates with RK4 at a fixed
dt = 0.1; this notebook uses SciPy's adaptive `odeint` (LSODA) with tight tolerances, so results agree
with the app within numerical tolerance.

In [ ]:
# 4. Run Simulation
t = np.linspace(0, 10, 101)
sol = odeint(model, y0, t, args=(params,), rtol=1e-8, atol=1e-10)

In [ ]:
# 5. Post-Processing (Calculate variables for plotting)
potential_adopters = sol[:, 0]
adopters = sol[:, 1]
total_population = np.full_like(t, params['total_population'])
adoption_fraction = np.full_like(t, params['adoption_fraction'])
ads_effectiveness = np.full_like(t, params['ads_effectiveness'])
contact_rate = np.full_like(t, params['contact_rate'])
word_of_mouth = (((potential_adopters * ((adopters / params['total_population']))) * params['contact_rate']) * params['adoption_fraction']) + np.zeros_like(t)
sales_from_ads = (potential_adopters * params['ads_effectiveness']) + np.zeros_like(t)
adoption_rate = (sales_from_ads + word_of_mouth) + np.zeros_like(t)

## Results

In [ ]:
# 6. Visualization
plt.figure(figsize=(10, 6))
plt.plot(t, adopters, label='adopters', linewidth=2)
plt.plot(t, adoption_rate, label='adoption_rate', linewidth=2)
plt.plot(t, potential_adopters, label='potential_adopters', linewidth=2)

plt.title('System Dynamics Simulation (Python)')
plt.xlabel('Time')
plt.ylabel('Value')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Experiment

Without advertising, why does adoption never start?

The cell below reruns the model with one change and compares each stock against the baseline.

In [ ]:
STOCKS = ['potential_adopters', 'adopters']

def simulate(param_changes=None, initial_changes=None):
    """Rerun the model with changed parameters and/or initial stock values."""
    p = {**params, **(param_changes or {})}
    y_init = list(y0)
    for stock, value in (initial_changes or {}).items():
        y_init[STOCKS.index(stock)] = value
    return odeint(model, y_init, t, args=(p,), rtol=1e-8, atol=1e-10)

scenario = simulate(param_changes={'ads_effectiveness': 0}, initial_changes={})

fig, axes = plt.subplots(len(STOCKS), 1, sharex=True, figsize=(10, 2.8 * len(STOCKS)))
axes = np.atleast_1d(axes)
for i, (ax, stock) in enumerate(zip(axes, STOCKS)):
    ax.plot(t, sol[:, i], label='Baseline', linewidth=2)
    ax.plot(t, scenario[:, i], '--', label='ads_effectiveness = 0', linewidth=2)
    ax.set_ylabel(stock)
    ax.grid(True, alpha=0.3)
axes[0].legend(loc='best')
axes[-1].set_xlabel('Time')
fig.suptitle('Baseline vs. scenario')
plt.tight_layout()
plt.show()

## Questions to explore

- Set ads_effectiveness to 0. Why does nothing happen?
- Which parameter moves the timing of the adoption peak most?